### IMPORTS

In [1]:
import nltk
import pandas as pd
import numpy as np
import string
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

### Preprocessing

In [ ]:
# Importing data as pandas DataFrame
df = pd.read_csv("train_submission.csv")

# Storing texts, labels and unique classes as lits
X = df['Text'].to_list()
y = df['Label'].to_list()
labels = df.groupby('Label').first().reset_index()['Label'].to_list()

# Splitting dataset as train and validation datasets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.1)

### Formating texts as in Cavnar and Trenkle (1994)

In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def preprocess_corpus(corpus):
    words = []
    
    for sentence in corpus:
        # Split sentence into words
        sentence = remove_punctuation(sentence)
        tokens = sentence.split()
        
        # Format each word with a leading space and 4 trailing spaces
        formatted_tokens = [f" {word}    " for word in tokens]
        
        # Add to final list
        words.extend(formatted_tokens)
    
    return words

Separating texts by language

In [4]:
texts_dict = {}

for label in tqdm(labels):
    X_lang = [X_train[i] for i,x in enumerate(y_train) if x == label]
    texts_dict[label] = X_lang

100%|██████████| 389/389 [00:01<00:00, 328.56it/s]


Ranking the K most frequent n-grams in every language

In [ ]:
def get_K_ngrams(corpus,K=300):
    X_preprocessed = preprocess_corpus(corpus)
    vectorizer = CountVectorizer(analyzer="char",ngram_range=(1,3),lowercase=True,max_features= K)
    ng_count = np.asarray(vectorizer.fit_transform(X_preprocessed).mean(axis=0)).flatten()
    ng = vectorizer.get_feature_names_out()

    #sort the N-grams by frequency in deescending order
    sorted_indices = np.argsort(-ng_count)
    ngram_df = pd.DataFrame({"ngram":ng[sorted_indices],"rank":list(range(len(ng)))})
    
    return ngram_df

In [6]:
n_grams = {}
vectorizer_dict = {}
for label in tqdm(labels):
    n_grams[label] = get_K_ngrams(texts_dict[label])

100%|██████████| 389/389 [00:18<00:00, 20.61it/s]


Computing the difference in ranks for most frequent ngrams lists

In [ ]:
def compute_rank_differences(list_one, list_two):
    # Create DataFrames for rankings
    df_one = pd.DataFrame({'word': list_one, 'rank_one': range(len(list_one))})
    df_two = pd.DataFrame({'word': list_two, 'rank_two': range(len(list_two))})

    # Merge on 'word' to align rankings
    merged = pd.merge(df_two, df_one, on='word', how='left')

    # Compute rank differences (NaN if word is missing from list_one)
    merged['rank_difference'] = merged['rank_two'] - merged['rank_one']

    return merged[['word', 'rank_difference']].fillna(len(merged))['rank_difference'].abs().sum()

11.0


Predicting labels for texts of the validation set

In [ ]:
y_pred = []
k = 1
for t in tqdm(X_test):
    # Computing the n_grams of the text
    guess = "ZZZ"
    n_grams_df = get_K_ngrams([t])
    best_sim = 1e10
    for label in labels:
        sim = compute_rank_differences(list(n_grams[label]["ngram"]),list(n_grams_df["ngram"]))
        if sim < best_sim:
            best_sim = sim
            guess = label

    y_pred.append(guess)

    if k % 100 == 0:
        print("Accuracy: ", accuracy_score(y_test[:k],y_pred))

    k += 1